# SealSign — эксперименты S05 в Colab

Выберите среду с GPU. Подготовьте доступный вам клон ветки `work` и приватный экспорт S04 в подключённом хранилище. Укажите пути к папке клона, экспорту и результатам. Результаты и веса сохраняются в папке `RUNS` между сессиями. Используйте Python 3.11 или 3.12.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path('/content/SealSign')
DATA = Path('/content/drive/MyDrive/SealSign/s04-expanded-v1')
RUNS = Path('/content/drive/MyDrive/SealSign/s05-runs')
CONFIG = REPO / 'configs/s05-yolo11n.json'
assert (REPO / 'scripts/train_s05.py').is_file()
assert (DATA / 'qc-report.json').is_file()
os.chdir(REPO)

def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True)

In [ ]:
run('-m', 'pip', 'install', '--index-url', 'https://download.pytorch.org/whl/cu128',
    'torch==2.7.1', 'torchvision==0.22.1')
run('-m', 'pip', 'install', 'ultralytics==8.3.228', 'Pillow==13.0.0')

In [ ]:
def train(experiment, resume=False):
    cmd = ['-m', 'scripts.train_s05', experiment, '--dataset', DATA,
           '--config', CONFIG, '--runs', RUNS]
    if resume:
        cmd += ['--resume', RUNS / experiment / 'weights/last.pt']
    run(*cmd)

train('E00')

Просмотрите `E00/results.csv`, `E00/args.yaml` и `E00/run-result.json` перед запуском baseline.

In [ ]:
train('E01')
# После прерывания незавершённого запуска вызовите train('E01', resume=True).

In [ ]:
run('-m', 'scripts.evaluate_s05', '--experiment', 'E01', '--split', 'val',
    '--data', DATA / 'dataset.yaml', '--config', CONFIG,
    '--weights', RUNS / 'E01/weights/best.pt', '--output', RUNS / 'E01/validation')

In [ ]:
run('-m', 'scripts.package_s05', '--runs', RUNS, '--experiments', 'E01',
    '--output', RUNS / 'bundle-v1')